# Vesper Capital — Wallet Search v3 (10k scale)

Scaled-up targeted search for skilled ETH wallets.

**Changes from v2:**
- `STAGE2_MAX_CANDIDATES = 10,000` — full scale run
- Tighter SPRT bounds [0.15, 7.0] vs [0.1, 10.0] — faster decisions
- Lower `STAGE2_MIN_TRADES_SPRT = 5` — SPRT triggers sooner
- `STAGE2_MIN_TRADES_ACCEPT = 10` — prevents zero-trade false positives
- `STAGE2_MAX_INCREMENTS = 4` — saves cost, 120 days max history
- `SKILLED_EDGE_THRESHOLD = 0.3%` — configurable via config cell
- Separate state file for 10k run (preserves 50-wallet results)
- Progress output every 10 wallets with running cost tracker
- Network expansion seeds from previously accepted HIGH quality wallets

**Estimated cost for full 10k run:**
- Stage 1: ~$0.07
- Stage 2 expected (~2 mean increments): ~$82
- Stage 2 worst case (4 increments): ~$164
- Run in multiple sessions of ~$9 each (restart-safe)

**Run plan:** set `MAX_CU_THIS_RUN = 20_000_000` (~$9/session), run overnight repeatedly until all 10k wallets processed.

## 1. Configuration

In [1]:
from claude_refactor_core_utils import *
import requests
import pandas as pd
import numpy as np
import json
import os
import pickle
from datetime import datetime, date, timedelta
from pathlib import Path
from collections import defaultdict
from scipy import stats as scipy_stats
from scipy.stats import norm as scipy_norm

ALCHEMY_API_KEY   = "8J0Ou3_XsuWVbOiZ0xpEvioGx7k0wab9"
ETHERSCAN_API_KEY = "GH6FVBANEIR1HS6PEN36XMYQRY2IDEAAXR"
BASE_URL          = f"https://eth-mainnet.g.alchemy.com/v2/{ALCHEMY_API_KEY}"

# ================================================================
# BUDGET CONTROLS
# ================================================================

# Stage 1
STAGE1_MAX_LOG_CALLS     = 2_000    # ~$0.068 — pulls ~100k-500k transfers
STAGE1_LOOKBACK_DAYS     = 90
STAGE1_MIN_TRADE_SIZE    = 0.1      # ETH
STAGE1_MAX_MEAN_TRADE    = 50.0     # ETH — exclude exchanges/whales
STAGE1_MIN_TRADE_COUNT   = 5
STAGE1_MAX_WALLETS       = 10_000   # forward to Stage 2

# Stage 2
STAGE2_MAX_CANDIDATES    = 10_000   # full run
STAGE2_SCORING_DAYS      = 30       # days per increment
STAGE2_MAX_INCREMENTS    = 4        # reduced from 6 — saves cost
STAGE2_MIN_TRADES_SPRT   = 5        # lowered from 10 — trigger SPRT sooner
STAGE2_MIN_TRADES_ACCEPT = 10       # min trades to be accepted as skilled

# SPRT parameters — tuned for faster decisions
# H0: wallet unskilled (earns baseline)
# H1: wallet skilled   (earns baseline + SKILLED_EDGE_THRESHOLD)
SKILLED_EDGE_THRESHOLD   = 0.3      # % above baseline to be considered skilled
STAGE2_SPRT_LOWER        = 0.15     # was 0.10 — abandon sooner
STAGE2_SPRT_UPPER        = 7.0      # was 10.0 — accept sooner

# Hard CU cap per run — restart-safe so can run multiple times
MAX_CU_THIS_RUN          = 10_000_000  # ~$9 per run

# Known exchange / MEV / protocol addresses to exclude
MEV_CONTRACTS = {
    '0x00000000003b3cc22af3ae1eac0440bcee416b40',
    '0x00000000219ab540356cbb839cbe05303d7705fa',
    '0x3f5ce5fbfe3e9af3971dd833d26ba9b5c936f0be',
    '0x28c6c06298d514db089934071355e5743bf21d60',
    '0x21a31ee1afc51d94c2efccaa2092ad1028285549',
    '0xdfd5293d8e347dfe59e90efd55b2956a1343963d',
    '0xae7ab96520de3a18e5e111b5eaab095312d7fe84',  # Lido stETH
    '0xbe0eb53f46cd790cd13851d5eff43d12404d33e8',  # Binance
    '0xf977814e90da44bfa03b6295a0616a897441acec',  # Binance
    '0x5a52e96bacdabb82fd05763e25335261b270efcb',  # Binance
    '0x8894e0a0c962cb723c1976a4421c95949be2d4e3',  # Binance
    '0x47ac0fb4f2d84898e4d9e7b4dab3c24507a6d503',  # Binance
}

# Paths
BASE_DIR   = Path('/home/colin/projects/blockchain_alpha')
SEARCH_DIR = BASE_DIR / 'data/wallet_search'
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_CSV   = SEARCH_DIR / 'stage1_candidates.csv'
ACCEPTED_CSV = SEARCH_DIR / 'accepted_wallets.csv'
STATE_PKL    = SEARCH_DIR / 'search_state_10k.pkl'  # separate state for 10k run

# CU/cost constants
cu_per_log_call     = 75
cu_per_balance     = 19      # theoretical per individual call
cu_per_balance_actual = 0.32  # empirical: 16000 CUs / 50 wallets / 1 increment
cu_rate_per_million = 0.45

# Cost estimates
s1_cu  = STAGE1_MAX_LOG_CALLS * cu_per_log_call
# With early stopping, expect ~2 mean increments per wallet
s2_cu_expected = int(STAGE2_MAX_CANDIDATES * 2 * STAGE2_SCORING_DAYS * 24 * cu_per_balance)
s2_cu_worst    = int(STAGE2_MAX_CANDIDATES * STAGE2_MAX_INCREMENTS * STAGE2_SCORING_DAYS * 24 * cu_per_balance)

print("=" * 65)
print("VESPER CAPITAL — WALLET SEARCH v3 (10k scale)")
print("=" * 65)
print()
print(f"Stage 1  ({STAGE1_MAX_LOG_CALLS} log calls):     ${s1_cu/1e6*cu_rate_per_million:.2f}")
print(f"Stage 2  expected (~2 inc avg): ${s2_cu_expected/1e6*cu_rate_per_million:.2f}")
print(f"Stage 2  worst case ({STAGE2_MAX_INCREMENTS} inc):    ${s2_cu_worst/1e6*cu_rate_per_million:.2f}")
print(f"Per-run cap:                    ${MAX_CU_THIS_RUN/1e6*cu_rate_per_million:.2f}")
print()
print(f"SPRT tuning (faster decisions):")
print(f"  Skilled edge threshold: +{SKILLED_EDGE_THRESHOLD}%/trade")
print(f"  SPRT bounds:            [{STAGE2_SPRT_LOWER}, {STAGE2_SPRT_UPPER}]  (was [0.1, 10.0])")
print(f"  Min trades for SPRT:    {STAGE2_MIN_TRADES_SPRT}  (was 10)")
print(f"  Min trades to accept:   {STAGE2_MIN_TRADES_ACCEPT}  (prevents zero-trade false positives)")
print()
print(f"Restart-safe: run multiple sessions of ${MAX_CU_THIS_RUN/1e6*cu_rate_per_million:.2f} each")
print(f"State file: {STATE_PKL}")


VESPER CAPITAL — WALLET SEARCH v3 (10k scale)

Stage 1  (2000 log calls):     $0.07
Stage 2  expected (~2 inc avg): $123.12
Stage 2  worst case (4 inc):    $246.24
Per-run cap:                    $4.50

SPRT tuning (faster decisions):
  Skilled edge threshold: +0.3%/trade
  SPRT bounds:            [0.15, 7.0]  (was [0.1, 10.0])
  Min trades for SPRT:    5  (was 10)
  Min trades to accept:   10  (prevents zero-trade false positives)

Restart-safe: run multiple sessions of $4.50 each
State file: /home/colin/projects/blockchain_alpha/data/wallet_search/search_state_10k.pkl


## 2. Block map

In [2]:
# ================================================================
# Load and extend block map
# ================================================================

print("Loading block map...")
bm_path = BASE_DIR / "data/hourly_block_map.csv"
if bm_path.exists():
    bm_df      = pd.read_csv(bm_path)
    bm_df["timestamp"] = pd.to_datetime(bm_df["timestamp"]).dt.floor("h")
    block_map  = dict(zip(bm_df["timestamp"], bm_df["block_number"].astype(int)))
    print(f"Loaded: {len(block_map):,} entries  ({min(block_map)} to {max(block_map)})")
else:
    block_map = {}
    print("No block map found")

def get_current_block():
    resp = requests.post(BASE_URL, json={
        "jsonrpc": "2.0", "method": "eth_blockNumber", "params": [], "id": 1
    }, timeout=10)
    return int(resp.json()["result"], 16)

current_block = get_current_block()
current_time  = pd.Timestamp.utcnow().floor("h").tz_localize(None)

if block_map:
    last_time  = max(block_map.keys())
    last_block = block_map[last_time]
    hours_gap  = int((current_time - last_time).total_seconds() / 3600)
    if hours_gap > 0:
        bph = (current_block - last_block) / hours_gap
        for h in range(1, hours_gap + 1):
            ts = last_time + pd.Timedelta(hours=h)
            block_map[ts] = int(last_block + bph * h)
        pd.DataFrame({
            "timestamp":    list(block_map.keys()),
            "block_number": list(block_map.values())
        }).sort_values("timestamp").to_csv(bm_path, index=False)
        print(f"Extended block map by {hours_gap:,} hours → {max(block_map)}")
else:
    bph = 300
    for h in range(24 * 200):
        ts = current_time - pd.Timedelta(hours=h)
        block_map[ts] = max(1, current_block - bph * h)
    print(f"Built block map: {len(block_map):,} entries")

# Pre-build block map DataFrame for fast slicing
bm_df_full = pd.DataFrame({
    "timestamp":    list(block_map.keys()),
    "block_number": list(block_map.values())
}).sort_values("timestamp").reset_index(drop=True)

print(f"Current block: {current_block:,}")


Loading block map...
Loaded: 29,308 entries  (2023-01-01 00:00:00 to 2026-05-06 03:00:00)
Current block: 25,033,360


## 3. Price data

In [3]:
# ================================================================
# Load price data for scoring window
# ================================================================

SCORING_END   = date.today() - timedelta(days=1)
SCORING_START = SCORING_END - timedelta(
    days=STAGE2_MAX_INCREMENTS * STAGE2_SCORING_DAYS + 30
)

print(f"Loading price data: {SCORING_START} to {SCORING_END}...")
df_price = load_eth_usd_cryptocompare(
    start=str(SCORING_START), end=str(SCORING_END), interval="1h"
)
df_price["date"] = pd.to_datetime(df_price["date"]).dt.floor("h")
df_price = df_price.sort_values("date").reset_index(drop=True)
df_price["fwd_return"] = (
    df_price["close"].shift(-24) - df_price["close"]
) / df_price["close"] * 100

BASELINE_RETURN = df_price["fwd_return"].dropna().mean()
SKILLED_RETURN  = BASELINE_RETURN + SKILLED_EDGE_THRESHOLD
RETURN_SIGMA    = df_price["fwd_return"].dropna().std()

print(f"Price data: {len(df_price):,} hours")
print(f"Baseline 24h return:  {BASELINE_RETURN:+.4f}%")
print(f"Skilled threshold:    {SKILLED_RETURN:+.4f}%  (baseline + {SKILLED_EDGE_THRESHOLD}%)")
print(f"Return std dev:       {RETURN_SIGMA:.4f}%")


Loading price data: 2025-12-05 to 2026-05-04...
Price data: 3,601 hours
Baseline 24h return:  -0.1395%
Skilled threshold:    +0.1605%  (baseline + 0.3%)
Return std dev:       3.3515%


## 4. Stage 1 — Candidate generation

In [4]:
# ================================================================
# Stage 1 — Candidate generation
# ================================================================

def fetch_large_eth_transfers(from_block, to_block, min_value_wei, max_calls=20):
    transfers  = []
    call_count = 0
    payload = {
        "jsonrpc": "2.0", "method": "alchemy_getAssetTransfers",
        "params": [{
            "fromBlock": hex(from_block), "toBlock": hex(to_block),
            "category": ["external"], "withMetadata": True,
            "excludeZeroValue": True, "maxCount": "0x3e8", "order": "asc",
        }], "id": 1
    }
    page_key = None
    while call_count < max_calls:
        if page_key:
            payload["params"][0]["pageKey"] = page_key
        resp  = requests.post(BASE_URL, json=payload, timeout=30)
        data  = resp.json().get("result", {})
        batch = data.get("transfers", [])
        call_count += 1
        for t in batch:
            val = float(t.get("value", 0) or 0)
            if val * 1e18 >= min_value_wei:
                transfers.append({
                    "from":      t.get("from", "").lower(),
                    "to":        t.get("to",   "").lower(),
                    "value_eth": val,
                    "timestamp": t.get("metadata", {}).get("blockTimestamp", ""),
                })
        page_key = data.get("pageKey")
        if not page_key:
            break
    return transfers, call_count

est_start_block = current_block - (STAGE1_LOOKBACK_DAYS * 7200)
min_value_wei   = int(STAGE1_MIN_TRADE_SIZE * 1e18)

print(f"Stage 1: pulling transfers from block {est_start_block:,} onwards")
print(f"Max calls: {STAGE1_MAX_LOG_CALLS}  |  Min trade: {STAGE1_MIN_TRADE_SIZE} ETH")
print()

all_transfers = []
total_calls   = 0
chunk_start   = est_start_block
CHUNK_SIZE    = 50_000

while chunk_start < current_block and total_calls < STAGE1_MAX_LOG_CALLS:
    chunk_end  = min(chunk_start + CHUNK_SIZE, current_block)
    transfers, calls = fetch_large_eth_transfers(
        chunk_start, chunk_end, min_value_wei,
        max_calls=min(20, STAGE1_MAX_LOG_CALLS - total_calls)
    )
    all_transfers.extend(transfers)
    total_calls   += calls
    chunk_start    = chunk_end + 1
    if total_calls % 100 == 0 or total_calls >= STAGE1_MAX_LOG_CALLS:
        print(f"  [{total_calls}/{STAGE1_MAX_LOG_CALLS} calls]  "
              f"transfers: {len(all_transfers):,}")

print(f"\nFetched {len(all_transfers):,} transfers  "
      f"cost: ${total_calls*cu_per_log_call/1e6*cu_rate_per_million:.3f}")


Stage 1: pulling transfers from block 24,385,360 onwards
Max calls: 2000  |  Min trade: 0.1 ETH

  [100/2000 calls]  transfers: 17,631
  [200/2000 calls]  transfers: 31,031

Fetched 43,929 transfers  cost: $0.009


In [5]:
# ================================================================
# Stage 1 filtering
# ================================================================

if not all_transfers:
    print("No transfers. Check API key.")
    candidates = pd.DataFrame()
else:
    df_t = pd.DataFrame(all_transfers)
    df_t["timestamp"] = pd.to_datetime(df_t["timestamp"], format="mixed", errors="coerce")
    df_t = df_t.dropna(subset=["timestamp"])
    df_t = df_t[~df_t["from"].isin(MEV_CONTRACTS)]
    df_t = df_t[~df_t["to"].isin(MEV_CONTRACTS)]

    buyers = df_t.groupby("to").agg(
        n_buys       = ("value_eth", "count"),
        total_eth_in = ("value_eth", "sum"),
        mean_eth_in  = ("value_eth", "mean"),
        std_eth_in   = ("value_eth", "std"),
    ).reset_index().rename(columns={"to": "address"})

    buyers = buyers[
        (buyers["n_buys"]      >= STAGE1_MIN_TRADE_COUNT) &
        (buyers["mean_eth_in"] >= STAGE1_MIN_TRADE_SIZE)  &
        (buyers["mean_eth_in"] <= STAGE1_MAX_MEAN_TRADE)  &
        (buyers["address"].str.len() == 42)
    ]

    buyers["trade_cv"]        = (buyers["std_eth_in"] /
                                  buyers["mean_eth_in"].clip(lower=0.001)
                                  ).fillna(999).clip(upper=10)
    buyers["candidate_score"] = (buyers["n_buys"] * buyers["mean_eth_in"] /
                                  (buyers["trade_cv"] + 0.1))
    buyers = buyers.sort_values("candidate_score", ascending=False)

    candidates = buyers.head(STAGE1_MAX_WALLETS).reset_index(drop=True)
    candidates.to_csv(STAGE1_CSV, index=False)

    print(f"Stage 1: {len(df_t):,} transfers → {len(buyers):,} wallets → "
          f"{len(candidates):,} candidates")
    print()
    print("Top 20 candidates:")
    print(candidates[["address","n_buys","mean_eth_in","trade_cv","candidate_score"
                       ]].head(20).round(3).to_string(index=False))


Stage 1: 42,326 transfers → 615 wallets → 615 candidates

Top 20 candidates:
                                   address  n_buys  mean_eth_in  trade_cv  candidate_score
0x4097dcf525f58acfc08952f184f3900961fa43e9      90        7.291     0.000         6561.469
0x44894aeee56c2dd589c1d5c8cb04b87576967f97      18       32.000     0.000         5760.000
0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511     581       45.338     8.291         3139.433
0xda7de2ecddfccc6c3af10108db212acbbf9ea83f       5       32.500     0.000         1625.000
0xbefd44b3834c8d45d711493b07eb7a887e12b2f3      22        6.354     0.004         1347.766
0xbeb5fc579115071764c7423a4f12edde41f106ed      54        7.562     0.228         1246.807
0x42655fc8443fb69e9ccdb94e89c1ee9317508a6d      16       28.938     0.294         1174.626
0x62425cd6bdcb6bfe51558ea465b063486b70dc9f     273       22.056     6.235          950.553
0x5b702b24b00a37b61ff9d71d91fed1a629edd1df      13       44.923     0.517          946.026
0xf30ba13e4b0

## 5. Network expansion

In [6]:
# ================================================================
# Network expansion from confirmed skilled wallets
# Run before Stage 2 to get highest-priority candidates
# ================================================================

existing_skilled = ["0x1b4b91bb747176451f0f7f73b636babafcf31cc4"]
if ACCEPTED_CSV.exists():
    prev_accepted = pd.read_csv(ACCEPTED_CSV)
    # Only use previously accepted wallets with real trade counts
    high_quality = prev_accepted[prev_accepted["n_trades"] >= 10]
    existing_skilled += high_quality["address"].tolist()

print(f"Network expansion from {len(existing_skilled)} seed wallets...")

def get_counterparties(address, alchemy_api_key, max_transfers=200):
    counterparties = defaultdict(float)
    for direction in ["fromAddress", "toAddress"]:
        payload = {
            "jsonrpc": "2.0", "method": "alchemy_getAssetTransfers",
            "params": [{direction: address, "category": ["external"],
                        "withMetadata": True, "maxCount": hex(max_transfers),
                        "order": "desc"}], "id": 1
        }
        resp = requests.post(
            f"https://eth-mainnet.g.alchemy.com/v2/{alchemy_api_key}",
            json=payload, timeout=15
        )
        for t in resp.json().get("result", {}).get("transfers", []):
            other = t.get(
                "to" if direction == "fromAddress" else "from", ""
            ).lower()
            val = float(t.get("value", 0) or 0)
            if other and other not in MEV_CONTRACTS and len(other) == 42:
                counterparties[other] += val
    return sorted(counterparties.items(), key=lambda x: x[1], reverse=True)

network_candidates = {}
for seed in existing_skilled:
    print(f"  {seed[:20]}...", end="  ")
    try:
        cp = get_counterparties(seed, ALCHEMY_API_KEY)
        for addr, eth in cp[:20]:
            if addr not in existing_skilled:
                network_candidates[addr] = network_candidates.get(addr, 0) + eth
        print(f"{len(cp)} counterparties found")
    except Exception as e:
        print(f"error: {e}")

network_df = pd.DataFrame([
    {"address": a, "total_eth_with_seeds": e}
    for a, e in sorted(network_candidates.items(), key=lambda x: x[1], reverse=True)
])

if len(network_df) > 0:
    network_df.to_csv(SEARCH_DIR / "network_candidates.csv", index=False)
    print(f"\n{len(network_df)} network candidates saved")
    print(network_df.head(10).to_string(index=False))
else:
    network_df = pd.DataFrame(columns=["address","total_eth_with_seeds"])
    print("No network candidates found")


Network expansion from 6 seed wallets...
  0x1b4b91bb747176451f...  50 counterparties found
  0x86a067030a9668c13f...  4 counterparties found
  0xb685760ebd368a891f...  181 counterparties found
  0xaacbf5550deaca37cf...  245 counterparties found
  0xf30ba13e4b04ce5dc4...  272 counterparties found
  0xae7ab96520de3a18e5...  180 counterparties found

102 network candidates saved
                                   address  total_eth_with_seeds
0xa03400e098f4421b34a3a44a1b4e571419517687           9867.791734
0xffd9dda6cf27311b580d16c6b474324424f9e1af            500.000000
0x78e6c4866d058ef05275e3aef3e1906da54525af            200.000000
0x5d0c7796aba12539c4caf0b9fa46ecc66c05ae9f            187.991200
0x8fca4ade3a517133ff23ca55cdaea29c78c990b8            185.554131
0xe7f1c19c6535b42351561b8fa4c8b43098952cf1            169.038931
0x7d652ab535f368c84ba375ee61c745fc05527e35            150.000000
0xb334a61a6209f14b5fa5f1684a4ed7621f66e1ef            139.221537
0x3eb609da99c07220b3d8efcce8b1e80be

## 6. Stage 2 helper functions

In [7]:
# ================================================================
# Stage 2 helper functions — v3 tuned for faster decisions
# ================================================================

def compute_skill_score(balance_df, price_df, min_delta=0.005):
    """Size-weighted forward return skill score."""
    bal = balance_df.copy()
    bal["delta_eth"] = bal["balance_eth"].diff()
    bal["action"]    = bal["delta_eth"].apply(
        lambda d: 1 if d > min_delta else (-1 if d < -min_delta else 0)
    )
    buys = bal[bal["action"] == 1].copy()
    if len(buys) < 3:
        return 0.0, 0, 1.0

    buys = buys.merge(
        price_df[["date","fwd_return"]].dropna(subset=["fwd_return"]),
        on="date", how="inner"
    )
    if len(buys) < 3:
        return 0.0, 0, 1.0

    baseline = price_df["fwd_return"].dropna().mean()
    weights  = np.nan_to_num(buys["delta_eth"].clip(lower=0).values, nan=0.0)
    mean_edge = (float(np.average(buys["fwd_return"], weights=weights) - baseline)
                 if weights.sum() > 0
                 else float(buys["fwd_return"].mean() - baseline))
    if np.isnan(mean_edge):
        mean_edge = 0.0
    _, p_val = scipy_stats.ttest_1samp(buys["fwd_return"].values, baseline)
    return mean_edge, len(buys), float(p_val)


def sprt_update(lr, observed, baseline, skilled, sigma,
                lower=STAGE2_SPRT_LOWER, upper=STAGE2_SPRT_UPPER):
    """Update SPRT likelihood ratio."""
    p0 = scipy_norm.pdf(observed, baseline, sigma)
    p1 = scipy_norm.pdf(observed, skilled,  sigma)
    new_lr = float(np.clip(lr * (p1 / p0) if p0 > 0 else lr * 10, 1e-6, 1e6))
    if new_lr >= upper:  return new_lr, "accept"
    if new_lr <= lower:  return new_lr, "abandon"
    return new_lr, "continue"


def safe_wtd_edge(all_returns, all_weights, baseline):
    """NaN-safe size-weighted edge computation."""
    if not all_returns:
        return 0.0
    r = np.array(all_returns)
    w = np.nan_to_num(np.array(all_weights), nan=0.0).clip(min=0)
    valid = ~np.isnan(r)
    r, w  = r[valid], w[valid]
    if len(r) == 0:
        return 0.0
    edge = float(np.average(r, weights=w) - baseline) if w.sum() > 0 else float(r.mean() - baseline)
    return 0.0 if np.isnan(edge) else edge


print("Stage 2 helper functions defined.")
print(f"SPRT: lower={STAGE2_SPRT_LOWER}  upper={STAGE2_SPRT_UPPER}  "
      f"skilled_threshold=+{SKILLED_EDGE_THRESHOLD}%")
cu_per_inc = STAGE2_SCORING_DAYS * 24 * cu_per_balance
print(f"Cost per increment per wallet: {cu_per_inc:,} CUs = "
      f"${cu_per_inc/1e6*cu_rate_per_million:.4f}")


Stage 2 helper functions defined.
SPRT: lower=0.15  upper=7.0  skilled_threshold=+0.3%
Cost per increment per wallet: 13,680 CUs = $0.0062


## 7. Stage 2 — Sequential scoring (10k scale)

**Restart-safe** — re-run this cell to continue from where it stopped.

Progress printed every 10 wallets showing: wallets processed, accepted count, cost so far, and mean increments per wallet.

Target mean increments < 2.5 — if all wallets are hitting max increments, tighten SPRT bounds further in cell 1.

In [8]:
# ================================================================
# Stage 2 main loop — 10k scale, restart-safe
#
# Key improvements over v2:
#   - Tighter SPRT bounds → earlier decisions
#   - Lower min_trades_sprt → SPRT triggers sooner
#   - Min trades to accept → no more zero-trade false positives
#   - Separate state file for 10k run
# ================================================================

# Build candidate list: network first, then Stage 1
network_path    = SEARCH_DIR / "network_candidates.csv"
network_wallets = (pd.read_csv(network_path)["address"].tolist()
                   if network_path.exists() else [])
stage1_wallets  = (pd.read_csv(STAGE1_CSV)["address"].tolist()
                   if STAGE1_CSV.exists() else [])

seen, wallet_list = set(), []
for addr in network_wallets + stage1_wallets:
    addr = addr.lower().strip()
    if addr not in seen and addr not in MEV_CONTRACTS:
        seen.add(addr)
        wallet_list.append(addr)
wallet_list = wallet_list[:STAGE2_MAX_CANDIDATES]

print(f"Candidate pool: {len(network_wallets)} network + "
      f"{len(stage1_wallets)} stage1 = {len(wallet_list)} unique")

# Warm start
if STATE_PKL.exists():
    with open(STATE_PKL, "rb") as f:
        state = pickle.load(f)
    n_done = len(state["results"])
    print(f"Warm start: {n_done} already processed  "
          f"({len(state['accepted'])} accepted, {len(state['abandoned'])} abandoned)")
else:
    state = {"results": {}, "accepted": [], "abandoned": [],
             "in_progress": {}, "total_cu": 0}

results, accepted, abandoned = state["results"], state["accepted"], state["abandoned"]
in_progress, total_cu        = state["in_progress"], state["total_cu"]
cu_this_run = 0

print(f"Budget this run: {MAX_CU_THIS_RUN:,} CUs  "
      f"(${MAX_CU_THIS_RUN/1e6*cu_rate_per_million:.2f})")
print()

for wallet_idx, address in enumerate(wallet_list):

    if address in accepted or address in abandoned:
        continue
    if cu_this_run >= MAX_CU_THIS_RUN:
        remaining = len([a for a in wallet_list[wallet_idx:]
                         if a not in accepted and a not in abandoned])
        print(f"\nBudget reached at wallet {wallet_idx+1}. "
              f"{remaining} remaining for next run.")
        break

    if address not in in_progress:
        in_progress[address] = {
            "increment": 0, "likelihood_ratio": 1.0,
            "all_returns": [], "all_weights": [], "n_trades_total": 0,
        }

    ws       = in_progress[address]
    decision = "continue"

    while ws["increment"] < STAGE2_MAX_INCREMENTS and decision == "continue":
        if cu_this_run >= MAX_CU_THIS_RUN:
            break

        inc_end   = SCORING_END - timedelta(days=STAGE2_SCORING_DAYS * ws["increment"])
        inc_start = inc_end    - timedelta(days=STAGE2_SCORING_DAYS)

        window_bm = bm_df_full[
            (bm_df_full["timestamp"] >= pd.Timestamp(inc_start)) &
            (bm_df_full["timestamp"] <= pd.Timestamp(inc_end))
        ].reset_index(drop=True)

        if len(window_bm) == 0:
            ws["increment"] += 1
            continue

        schedule = pd.DataFrame({
            "address": address,
            "date":    pd.date_range(start=inc_start, end=inc_end, freq="h")
        })

        try:
            bal_raw = fetch_balances_for_schedule(
                schedule=schedule, block_df=window_bm,
                alchemy_api_key=ALCHEMY_API_KEY,
                batch_size=100, max_concurrency=20, rps=25.0,
                progress_every=99999, block_date_col="timestamp",
                label=f"{address[:10]}-i{ws['increment']+1}",
            )
        except Exception as e:
            ws["increment"] += 1
            continue

        cu_this_run += int(cu_per_balance_actual * 24 * STAGE2_SCORING_DAYS)
        total_cu    += len(schedule) * cu_per_balance
        ws["increment"] += 1

        if bal_raw is None or bal_raw.empty:
            continue

        bal_raw["date"]        = pd.to_datetime(bal_raw["date"], format="mixed").dt.floor("h")
        bal_raw["balance_eth"] = pd.to_numeric(bal_raw["balance_wei"], errors="coerce").fillna(0) / 1e18
        bal_df = bal_raw[["date","balance_eth"]].sort_values("date").reset_index(drop=True)

        # Skill score for this increment
        edge_inc, n_trades_inc, _ = compute_skill_score(bal_df, df_price)
        ws["n_trades_total"] += n_trades_inc

        # Collect trade-level data for SPRT
        bal_df["delta_eth"] = bal_df["balance_eth"].diff()
        bal_df["action"]    = bal_df["delta_eth"].apply(lambda d: 1 if d > 0.005 else 0)
        buy_rows = bal_df[bal_df["action"] == 1][["date","delta_eth"]].copy()
        buy_rows = buy_rows.merge(
            df_price[["date","fwd_return"]].dropna(subset=["fwd_return"]),
            on="date", how="inner"
        )
        ws["all_returns"].extend(buy_rows["fwd_return"].tolist())
        ws["all_weights"].extend(
            buy_rows["delta_eth"].fillna(0.0).clip(lower=0).tolist()
        )

        # Apply SPRT once enough trades observed
        if ws["n_trades_total"] >= STAGE2_MIN_TRADES_SPRT:
            for ret in buy_rows["fwd_return"].tolist():
                ws["likelihood_ratio"], decision = sprt_update(
                    ws["likelihood_ratio"], ret,
                    BASELINE_RETURN, SKILLED_RETURN, RETURN_SIGMA
                )
                if decision != "continue":
                    break

    # ── Final decision ────────────────────────────────────────
    if decision == "continue":
        # Max increments reached without SPRT decision
        wtd_edge = safe_wtd_edge(ws["all_returns"], ws["all_weights"], BASELINE_RETURN)
        if ws["n_trades_total"] >= STAGE2_MIN_TRADES_ACCEPT and wtd_edge > 0.2:
            decision = "accept"
        else:
            decision = "abandon"

    wtd_edge = safe_wtd_edge(ws["all_returns"], ws["all_weights"], BASELINE_RETURN)

    # SPRT accepted but insufficient trades → downgrade to abandon
    if decision == "accept" and ws["n_trades_total"] < STAGE2_MIN_TRADES_ACCEPT:
        decision = "abandon_insufficient_trades"
        abandoned.append(address)
    elif decision == "accept":
        accepted.append(address)
    else:
        abandoned.append(address)

    results[address] = {
        "address":          address,
        "decision":         decision,
        "increments_used":  ws["increment"],
        "n_trades":         ws["n_trades_total"],
        "weighted_edge":    wtd_edge,
        "likelihood_ratio": ws["likelihood_ratio"],
        "cu_used":          ws["increment"] * STAGE2_SCORING_DAYS * 24 * cu_per_balance,
    }

    # Progress output every 10 wallets
    if (wallet_idx + 1) % 10 == 0 or decision == "accept":
        accepted_count  = len([r for r in results.values() if r["decision"]=="accept"])
        abandoned_count = len([r for r in results.values() if "abandon" in r["decision"]])
        pct_done = (wallet_idx + 1) / len(wallet_list) * 100
        cost_so_far = cu_this_run / 1e6 * cu_rate_per_million
        tag = "✓ ACCEPT" if decision == "accept" else ""
        print(f"[{wallet_idx+1:5d}/{len(wallet_list)}  {pct_done:5.1f}%  "
              f"${cost_so_far:.3f}]  "
              f"accepted={accepted_count}  "
              f"abandoned={abandoned_count}  "
              f"mean_inc={np.mean([r['increments_used'] for r in results.values()]):.1f}  "
              f"{tag}")

    # Save state every 50 wallets
    if (wallet_idx + 1) % 50 == 0:
        state.update({"results": results, "accepted": accepted,
                      "abandoned": abandoned, "in_progress": in_progress,
                      "total_cu": total_cu})
        with open(STATE_PKL, "wb") as f:
            pickle.dump(state, f)

# Final save
state.update({"results": results, "accepted": accepted,
              "abandoned": abandoned, "in_progress": in_progress,
              "total_cu": total_cu})
with open(STATE_PKL, "wb") as f:
    pickle.dump(state, f)

print(f"\nRun complete.")
print(f"  Processed this run: {len(results)}")
print(f"  Accepted:           {len(accepted)}")
print(f"  CUs this run:       {cu_this_run:,}")
print(f"  Cost this run:      ${cu_this_run/1e6*cu_rate_per_million:.3f}")


Candidate pool: 102 network + 615 stage1 = 706 unique
Budget this run: 10,000,000 CUs  ($4.50)

[0xa03400e0-i1] 31/31 batches (100.0%)
[0xa03400e0-i2] 31/31 batches (100.0%)
[0xa03400e0-i3] 31/31 batches (100.0%)
[0xa03400e0-i4] 31/31 batches (100.0%)
[    1/706    0.1%  $0.000]  accepted=1  abandoned=0  mean_inc=4.0  ✓ ACCEPT
[0xffd9dda6-i1] 31/31 batches (100.0%)
[0xffd9dda6-i2] 31/31 batches (100.0%)
[0xffd9dda6-i3] 31/31 batches (100.0%)
[0xffd9dda6-i4] 31/31 batches (100.0%)
[0x78e6c486-i1] 31/31 batches (100.0%)
[0x78e6c486-i2] 31/31 batches (100.0%)
[0x78e6c486-i3] 31/31 batches (100.0%)
[0x78e6c486-i4] 31/31 batches (100.0%)
[0x5d0c7796-i1] 31/31 batches (100.0%)
[0x5d0c7796-i2] 31/31 batches (100.0%)
[0x5d0c7796-i3] 31/31 batches (100.0%)
[0x5d0c7796-i4] 31/31 batches (100.0%)
[0x8fca4ade-i1] 31/31 batches (100.0%)
[0x8fca4ade-i2] 31/31 batches (100.0%)
[0x8fca4ade-i3] 31/31 batches (100.0%)
[0x8fca4ade-i4] 31/31 batches (100.0%)
[    5/706    0.7%  $0.002]  accepted=2  abando

## 8. Results

In [9]:
# ================================================================
# Results analysis
# ================================================================

results_df = pd.DataFrame(list(results.values()))
if results_df.empty:
    print("No results yet.")
else:
    accepted_df  = results_df[results_df["decision"]=="accept"].sort_values(
        "weighted_edge", ascending=False
    )

    print("=" * 65)
    print("STAGE 2 RESULTS SUMMARY")
    print("=" * 65)
    print(f"Total processed:      {len(results_df):,}")
    print(f"Accepted (skilled):   {len(accepted_df)}")
    print(f"Abandoned:            {len(results_df)-len(accepted_df)}")
    print(f"Hit rate:             {len(accepted_df)/len(results_df)*100:.1f}%")
    print()

    mean_inc = results_df["increments_used"].mean()
    tot_cu   = results_df["cu_used"].sum()
    print(f"Mean increments/wallet: {mean_inc:.2f}  (target: < 2.5 with SPRT)")
    print(f"Total CU used:          {tot_cu:,}")
    print(f"Total cost:             ${tot_cu/1e6*cu_rate_per_million:.3f}")
    if len(results_df) > 0:
        print(f"Cost per wallet:        ${tot_cu/len(results_df)/1e6*cu_rate_per_million:.4f}")
    print()

    print("Early stopping:")
    for inc in range(1, STAGE2_MAX_INCREMENTS + 1):
        n   = (results_df["increments_used"] == inc).sum()
        pct = n / len(results_df) * 100
        bar = "█" * int(pct / 2)
        print(f"  Stopped at {inc}: {n:4d} ({pct:4.1f}%)  {bar}")
    print()

    print("Weighted edge distribution (all wallets):")
    print(results_df["weighted_edge"].describe().round(4))
    print()

    if len(accepted_df) > 0:
        print("Accepted wallets:")
        print(accepted_df[[
            "address","weighted_edge","n_trades","increments_used","likelihood_ratio"
        ]].round(4).to_string(index=False))
        accepted_df.to_csv(ACCEPTED_CSV, index=False)
        print(f"\nSaved: {ACCEPTED_CSV}")

        # Quality tiers
        high = accepted_df[
            (accepted_df["n_trades"] >= 20) & (accepted_df["weighted_edge"] > 0.5)
        ]
        med  = accepted_df[
            (accepted_df["n_trades"] >= 10) & (accepted_df["n_trades"] < 20)
        ]
        low  = accepted_df[accepted_df["n_trades"] < 10]

        print()
        print(f"Quality tiers:")
        print(f"  HIGH (>=20 trades, >0.5% edge): {len(high)}")
        print(f"  MED  (10-19 trades):            {len(med)}")
        print(f"  LOW  (<10 trades):              {len(low)}")
        print()
        print("Next step: run hourly fetch on HIGH + MED quality wallets")
        print("Then add to wallet pool and retrain walk-forward model.")


STAGE 2 RESULTS SUMMARY
Total processed:      706
Accepted (skilled):   113
Abandoned:            593
Hit rate:             16.0%

Mean increments/wallet: 4.00  (target: < 2.5 with SPRT)
Total CU used:          38,632,320
Total cost:             $17.385
Cost per wallet:        $0.0246

Early stopping:
  Stopped at 1:    0 ( 0.0%)  
  Stopped at 2:    0 ( 0.0%)  
  Stopped at 3:    0 ( 0.0%)  
  Stopped at 4:  706 (100.0%)  ██████████████████████████████████████████████████

Weighted edge distribution (all wallets):
count    706.0000
mean      -0.1404
std        1.6298
min      -12.7842
25%       -0.4238
50%        0.0000
75%        0.1998
max        9.4563
Name: weighted_edge, dtype: float64

Accepted wallets:
                                   address  weighted_edge  n_trades  increments_used  likelihood_ratio
0x90843bb8e525ee77aea03b97a8826b6ee1bab43e         5.3083        11                4            0.7062
0x50553500f157e5c3b0bf4ddfd64dd8229492883d         5.1117        83       